# SC-8b - Developpement Smart Contracts Assiste par LLM

**Navigation** : [Index](../README.md) | [<< Account Abstraction](SC-8-Account-Abstraction.ipynb) | [Foundry Basics >>](../03-Foundry-Testing/SC-9-Foundry-Basics.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Utiliser les **LLMs** (Claude, GPT) pour generer du code Solidity
2. Appliquer des **patterns de prompting** efficaces pour les smart contracts
3. Exploiter les LLMs pour la **detection de vulnerabilites**
4. Generer automatiquement la **documentation NatSpec**
5. Integrer l'API **Anthropic Claude** dans un workflow de developpement
6. Utiliser un **LLM local** (Qwen) comme alternative open-source

### Prerequis

- Notebooks SC-1 a SC-4 (fondements Solidity)
- Cle API OpenAI ou Anthropic (optionnel pour tests)
- Python 3.10+ avec `openai`, `anthropic`, `requests`

### Duree estimee : 45 minutes

---

## Introduction

Les **Large Language Models** ont revolutionne le developpement de smart contracts. Ils peuvent :

- **Generer du code** a partir de specifications en langage naturel
- **Auditer** des contrats pour detecter des vulnerabilites
- **Documenter** automatiquement avec des commentaires NatSpec
- **Expliquer** des concepts complexes Solidity

### Pourquoi utiliser les LLMs pour Solidity ?

| Avantage | Description |
|----------|-------------|
| **Rapidite** | Prototypage en minutes, pas en heures |
| **Apprentissage** | Explications contextuelles du code genere |
| **Securite** | Detection precoce de patterns vulnerables |
| **Documentation** | NatSpec genere automatiquement |

### Limitations importantes

> **Attention** : Les LLMs peuvent generer du code contenant des bugs ou vulnerabilites.
> Toujours auditer, tester et verifier formellement le code genere avant deploiement en production.

## 1. Configuration et Imports

In [ ]:
# Imports standards
import os
import json
import re
from typing import Dict, List, Optional, Any
from dataclasses import dataclass
from pathlib import Path

# API clients - Optionnels avec graceful fallback
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    HAS_OPENAI = False
    OpenAI = None

try:
    from anthropic import Anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    Anthropic = None

# Charger les variables d'environnement
try:
    from dotenv import load_dotenv
    load_dotenv()
    HAS_DOTENV = True
except ImportError:
    HAS_DOTENV = False

print("Imports charges avec succes")
print(f"OpenAI disponible: {HAS_OPENAI}")
print(f"Anthropic disponible: {HAS_ANTHROPIC}")

In [ ]:
# Configuration des clients API
@dataclass
class LLMConfig:
    """Configuration pour les appels LLM"""
    openai_api_key: str = os.getenv("OPENAI_API_KEY", "") if HAS_DOTENV else ""
    anthropic_api_key: str = os.getenv("ANTHROPIC_API_KEY", "") if HAS_DOTENV else ""
    model_openai: str = "gpt-4o"
    model_anthropic: str = "claude-sonnet-4-6"
    temperature: float = 0.7
    max_tokens: int = 2000

config = LLMConfig()

# Initialiser les clients (si disponibles et configures)
openai_client = None
anthropic_client = None

if HAS_OPENAI and config.openai_api_key:
    try:
        openai_client = OpenAI(api_key=config.openai_api_key)
        print("Client OpenAI initialise")
    except Exception as e:
        print(f"Erreur initialisation OpenAI: {e}")

if HAS_ANTHROPIC and config.anthropic_api_key:
    try:
        anthropic_client = Anthropic(api_key=config.anthropic_api_key)
        print("Client Anthropic initialise")
    except Exception as e:
        print(f"Erreur initialisation Anthropic: {e}")

print(f"\nStatut:")
print(f"  OpenAI client: {'Configure' if openai_client else 'Non configure'}")
print(f"  Anthropic client: {'Configure' if anthropic_client else 'Non configure'}")

### Interpretation

La configuration utilise des variables d'environnement pour les cles API. Le notebook fonctionne meme sans API configuree, en utilisant des exemples de mock responses.

**Pour configurer vos cles** :
1. Creez un fichier `.env` dans le repertoire
2. Ajoutez : `OPENAI_API_KEY=sk-...` ou `ANTHROPIC_API_KEY=sk-ant-...`

## 2. Prompt Patterns pour Solidity

Le prompting efficace pour Solidity necessite structure et precision. Voici les templates de base.

In [ ]:
# Templates de prompts pour Solidity

SOLIDITY_PROMPTS = {
    "contract_generation": """Tu es un expert en developpement de smart contracts Solidity.

Genere un contrat Solidity complet pour le cas d'usage suivant:
{description}

Contraintes:
- Version Solidity: ^0.8.20
- Inclure les imports OpenZeppelin si necessaire
- Ajouter les commentaires NatSpec complets
- Implementer les checks de securite (ReentrancyGuard, etc.)
- Utiliser les patterns les plus recents et securises

Fournit uniquement le code Solidity, sans explications supplementaires.
""",

    "audit": """Tu es un auditeur de securite smart contracts expert.

Analyse ce contrat Solidity pour les vulnerabilites potentielles:
```solidity
{code}
```

Identifie:
1. Vulnerabilites connues (reentrancy, overflow, etc.)
2. Problems de gas optimization
3. Centralisation risks
4. Best practices non respectees

Format de sortie:
- SEVERITE (CRITICAL/HIGH/MEDIUM/LOW)
- Description du probleme
- Ligne concernee
- Recommandation de correction
""",

    "natspec": """Genere la documentation NatSpec complete pour ce contrat Solidity:
```solidity
{code}
```

Inclus pour chaque fonction:
- @title et @description pour le contrat
- @notice pour l'utilisateur
- @dev pour les details techniques
- @param pour chaque parametre
- @return pour les valeurs de retour

Retourne le code avec les commentaires NatSpec ajoutes.
""",

    "explain": """Explique ce code Solidity de maniere pedagogique:
```solidity
{code}
```

Structure ton explication:
1. Objectif global du contrat
2. Variables d'etat et leur role
3. Fonctions principales
4. Patterns de securite utilises
5. Cas d'usage typiques
"""
}

print("Templates de prompts definis:")
for name in SOLIDITY_PROMPTS:
    print(f"  - {name}")

### 2.1 Bonnes pratiques de prompting

| Pratique | Exemple | Impact |
|----------|---------|--------|
| **Specifier la version** | "Solidity ^0.8.20" | Code compatible |
| **Inclure OpenZeppelin** | "Utilise IERC20 de OpenZeppelin" | Standards respectes |
| **Demander NatSpec** | "Ajoute les commentaires NatSpec" | Documentation auto |
| **Contraintes de securite** | "Utilise ReentrancyGuard" | Code plus sur |
| **Format de sortie** | "Retourne uniquement le code" | Parsing facilite |

## 3. Generation de Contrats via API

Implementons un generateur de smart contracts utilisant l'API Anthropic Claude.

In [ ]:
def extract_solidity_code(response_text: str) -> str:
    """Extrait le code Solidite d'une reponse LLM"""
    # Chercher les blocs de code Solidity
    pattern = r'```solidity\s*\n(.*?)```'
    matches = re.findall(pattern, response_text, re.DOTALL)
    
    if matches:
        return matches[0].strip()
    
    # Fallback: chercher n'importe quel bloc de code
    pattern = r'```\s*\n(.*?)```'
    matches = re.findall(pattern, response_text, re.DOTALL)
    
    if matches:
        return matches[0].strip()
    
    return response_text


def generate_contract(
    description: str,
    client_type: str = "anthropic",
    temperature: float = 0.7
) -> tuple:
    """
    Genere un contrat Solidity via LLM.
    Retourne (code_solide, reponse_complete)
    """
    prompt = SOLIDITY_PROMPTS["contract_generation"].format(description=description)
    
    if client_type == "anthropic" and anthropic_client:
        try:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}]
            )
            full_response = response.content[0].text
            solidity_code = extract_solidity_code(full_response)
            return solidity_code, full_response
        except Exception as e:
            return None, f"Erreur API Anthropic: {e}"
    
    elif client_type == "openai" and openai_client:
        try:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}]
            )
            full_response = response.choices[0].message.content
            solidity_code = extract_solidity_code(full_response)
            return solidity_code, full_response
        except Exception as e:
            return None, f"Erreur API OpenAI: {e}"
    
    else:
        # Mock response quand pas d'API
        mock_response = f"""```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Contrat genere par LLM (mock)
/// @dev Ceci est un exemple de mock - configurez l'API pour du vrai code
contract GeneratedContract {{
    // Implementation basee sur: {description[:50]}...
    
    address public owner;
    
    constructor() {{
        owner = msg.sender;
    }}
    
    modifier onlyOwner() {{
        require(msg.sender == owner, "Not owner");
        _;
    }}
}}
```"""
        return extract_solidity_code(mock_response), mock_response


print("Fonction generate_contract definie")

In [ ]:
# Exemple: Generer un contrat simple
description = """
Un contrat de stockage simple qui permet:
- Stocker une valeur uint256
- Recuperer la valeur stockee
- Seul le owner peut modifier la valeur
- Emettre un event quand la valeur change
"""

print("Generation d'un contrat de stockage...")
print(f"Description: {description.strip()}")
print("\n" + "="*60)

code, response = generate_contract(description)

if code:
    print("CONTRAT GENERE:")
    print("="*60)
    print(code)
else:
    print(f"Erreur: {response}")

### Interpretation

Le contrat genere devrait inclure :

| Element | Description |
|---------|-------------|
| **SPDX License** | Identifiant de licence MIT |
| **Pragma** | Version Solidity ciblee |
| **NatSpec** | Documentation des fonctions |
| **Event** | ValueChanged pour tracking |
| **Modifier** | onlyOwner pour access control |

## 4. Audit de Securite avec LLM

Utilisons les LLMs pour detecter les vulnerabilites dans les smart contracts.

In [ ]:
def audit_contract(code: str, client_type: str = "anthropic") -> str:
    """
    Audite un contrat Solidity pour les vulnerabilites.
    """
    prompt = SOLIDITY_PROMPTS["audit"].format(code=code)
    
    if client_type == "anthropic" and anthropic_client:
        try:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        except Exception as e:
            return f"Erreur API: {e}"
    
    elif client_type == "openai" and openai_client:
        try:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Erreur API: {e}"
    
    else:
        # Mock audit response
        return """## Audit de Securite (Mock Response)

### VULNERABILITES DETECTEES:

**CRITICAL - Reentrancy Risk**
- Ligne: N/A
- Description: Ce mock ne peut pas analyser le code reel
- Recommandation: Configurez l'API pour un audit reel

**HIGH - Centralisation Risk**
- Ligne: constructor
- Description: Le pattern owner unique cree un point de centralisation
- Recommandation: Considerer un systeme multi-sig pour la production

### RECOMMANDATIONS:
1. Utiliser ReentrancyGuard de OpenZeppelin
2. Ajouter des tests avec Foundry
3. Effectuer une verification formelle avec Certora
"""


print("Fonction audit_contract definie")

In [ ]:
# Exemple: Auditer un contrat vulnerables
vulnerable_contract = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract VulnerableVault {
    mapping(address => uint256) public balances;
    
    function deposit() external payable {
        balances[msg.sender] += msg.value;
    }
    
    function withdraw() external {
        uint256 amount = balances[msg.sender];
        // VULNERABILITE: appel externe avant mise a jour
        (bool success, ) = msg.sender.call{value: amount}("");
        require(success, "Transfer failed");
        balances[msg.sender] = 0;
    }
}
"""

print("=== AUDIT DE SECURITE ===")
print("Contrat analyse: VulnerableVault")
print("\n" + "="*60)

audit_result = audit_contract(vulnerable_contract)
print(audit_result)

### Interpretation

L'audit LLM devrait identifier la **reentrancy vulnerability** dans le contrat ci-dessus :

```
1. Utilisateur A appelle withdraw()
2. Le contrat envoie les fonds a A
3. A est un contrat malveillant qui reappelle withdraw()
4. Le solde n'a pas encore ete mis a jour (toujours > 0)
5. Le contrat envoie les fonds une deuxieme fois !
```

**Correction recommandee** : Pattern Checks-Effects-Interactions ou ReentrancyGuard.

## 5. Generation de Documentation NatSpec

La documentation NatSpec (Ethereum Natural Language Specification) est essentielle pour les smart contracts.

In [ ]:
def generate_natspec(code: str, client_type: str = "anthropic") -> str:
    """
    Genere la documentation NatSpec pour un contrat.
    """
    prompt = SOLIDITY_PROMPTS["natspec"].format(code=code)
    
    if client_type == "anthropic" and anthropic_client:
        try:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        except Exception as e:
            return f"Erreur API: {e}"
    
    elif client_type == "openai" and openai_client:
        try:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Erreur API: {e}"
    
    else:
        # Mock NatSpec generation
        return f"""```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title SimpleStorage
/// @author LLM Generated
/// @notice Un contrat simple pour stocker et recuperer une valeur
/// @dev Utilise le pattern Ownable pour le controle d'acces
contract SimpleStorage {{
    /// @notice La valeur stockee dans le contrat
    /// @dev Visible publiquement, modifiable uniquement par le owner
    uint256 public storedValue;
    
    /// @notice Adresse du proprietaire du contrat
    address public owner;
    
    /// @notice Emet quand la valeur est modifiee
    /// @param oldValue L'ancienne valeur
    /// @param newValue La nouvelle valeur
    /// @param changedBy L'adresse qui a effectue le changement
    event ValueChanged(uint256 oldValue, uint256 newValue, address changedBy);
    
    /// @notice Initialise le contrat avec le sender comme owner
    constructor() {{
        owner = msg.sender;
    }}
    
    /// @notice Modifie la valeur stockee
    /// @dev Seul le owner peut appeler cette fonction
    /// @param newValue La nouvelle valeur a stocker
    function setValue(uint256 newValue) external {{
        require(msg.sender == owner, "Only owner can set value");
        uint256 oldValue = storedValue;
        storedValue = newValue;
        emit ValueChanged(oldValue, newValue, msg.sender);
    }}
    
    /// @notice Recupere la valeur stockee
    /// @return La valeur actuellement stockee
    function getValue() external view returns (uint256) {{
        return storedValue;
    }}
}}
```"""


print("Fonction generate_natspec definie")

In [ ]:
# Exemple: Ajouter NatSpec a un contrat sans documentation
undocumented_contract = """
pragma solidity ^0.8.20;

contract Token {
    string public name = "MyToken";
    string public symbol = "MTK";
    uint8 public decimals = 18;
    uint256 public totalSupply;
    mapping(address => uint256) public balanceOf;
    
    event Transfer(address indexed from, address indexed to, uint256 value);
    
    constructor(uint256 _initialSupply) {
        totalSupply = _initialSupply * 10 ** uint256(decimals);
        balanceOf[msg.sender] = totalSupply;
    }
    
    function transfer(address _to, uint256 _value) public returns (bool success) {
        require(balanceOf[msg.sender] >= _value, "Insufficient balance");
        balanceOf[msg.sender] -= _value;
        balanceOf[_to] += _value;
        emit Transfer(msg.sender, _to, _value);
        return true;
    }
}
"""

print("=== GENERATION NATSPEC ===")
print("Contrat original sans documentation")
print("\n" + "="*60)

documented = generate_natspec(undocumented_contract)
print(documented)

### Interpretation

La documentation NatSpec generee inclut les tags standards :

| Tag | Usage |
|-----|-------|
| `@title` | Nom du contrat |
| `@author` | Auteur du code |
| `@notice` | Description utilisateur |
| `@dev` | Details techniques |
| `@param` | Description des parametres |
| `@return` | Description de la valeur de retour |
| `@event` | Description des evenements |

## 6. Integration avec l'API Anthropic Claude

Exemple d'integration complete avec Claude Sonnet 4.6 pour un workflow de developpement.

In [ ]:
class SolidityLLMAssistant:
    """
    Assistant LLM pour le developpement de smart contracts.
    """
    
    def __init__(self, client_type: str = "anthropic"):
        self.client_type = client_type
        self.client = anthropic_client if client_type == "anthropic" else openai_client
        self.model = config.model_anthropic if client_type == "anthropic" else config.model_openai
    
    def generate(
        self,
        description: str,
        temperature: float = 0.7,
        max_tokens: int = 2000
    ) -> Dict[str, Any]:
        """Genere un contrat complet avec metadonnees"""
        code, response = generate_contract(description, self.client_type)
        
        return {
            "code": code,
            "full_response": response,
            "model": self.model,
            "provider": self.client_type,
            "success": code is not None
        }
    
    def audit(self, code: str) -> Dict[str, Any]:
        """Audite un contrat pour les vulnerabilites"""
        report = audit_contract(code, self.client_type)
        
        return {
            "report": report,
            "model": self.model,
            "provider": self.client_type
        }
    
    def document(self, code: str) -> Dict[str, Any]:
        """Ajoute la documentation NatSpec"""
        documented = generate_natspec(code, self.client_type)
        
        return {
            "documented_code": extract_solidity_code(documented),
            "full_response": documented,
            "model": self.model,
            "provider": self.client_type
        }
    
    def full_workflow(self, description: str) -> Dict[str, Any]:
        """
        Workflow complet: generation -> audit -> documentation
        """
        # 1. Generer
        gen_result = self.generate(description)
        if not gen_result["success"]:
            return {"error": "Generation failed", "details": gen_result}
        
        # 2. Auditer
        audit_result = self.audit(gen_result["code"])
        
        # 3. Documenter
        doc_result = self.document(gen_result["code"])
        
        return {
            "original_code": gen_result["code"],
            "documented_code": doc_result["documented_code"],
            "audit_report": audit_result["report"],
            "model": self.model,
            "provider": self.client_type
        }


# Initialiser l'assistant
assistant = SolidityLLMAssistant(client_type="anthropic" if anthropic_client else "openai")
print(f"Assistant initialise avec {assistant.client_type} ({assistant.model})")

In [ ]:
# Demo du workflow complet
print("=== WORKFLOW COMPLET LLM ===")
print("\nDescription: Un compteur avec increment et decrement")
print("\n" + "="*60)

result = assistant.full_workflow("""
Un contrat Counter qui permet:
- Incrementer un compteur
- Decrementer le compteur (minimum 0)
- Lire la valeur actuelle
- Seul le owner peut incrementer/decrementer
""")

if "error" not in result:
    print("CODE GENERE:")
    print("-" * 40)
    print(result["original_code"][:500] + "..." if len(result["original_code"] or "") > 500 else result["original_code"])
    
    print("\n" + "="*60)
    print("AUDIT (extrait):")
    print("-" * 40)
    print(result["audit_report"][:400] + "..." if len(result["audit_report"] or "") > 400 else result["audit_report"])
else:
    print(f"Erreur: {result['error']}")

## 7. LLM Local avec Qwen (Optionnel)

Pour les cas ou les API cloud ne sont pas disponibles ou souhaites, on peut utiliser un LLM local via un endpoint.

In [ ]:
import requests

class LocalLLMAssistant:
    """
    Assistant utilisant un LLM local (ex: Qwen via vLLM ou Ollama).
    """
    
    def __init__(
        self,
        endpoint: str = "http://localhost:11434/api/generate",
        model: str = "qwen2.5-coder:7b"
    ):
        self.endpoint = endpoint
        self.model = model
        self.available = self._check_availability()
    
    def _check_availability(self) -> bool:
        """Verifie si le serveur local est disponible"""
        try:
            response = requests.get(
                self.endpoint.replace("/api/generate", ""),
                timeout=2
            )
            return response.status_code == 200
        except:
            return False
    
    def generate(self, prompt: str, temperature: float = 0.7) -> str:
        """Genere une reponse via le LLM local"""
        if not self.available:
            return "LLM local non disponible. Demarrez Ollama ou vLLM."
        
        try:
            # Format Ollama
            response = requests.post(
                self.endpoint,
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "temperature": temperature,
                    "stream": False
                },
                timeout=60
            )
            
            if response.status_code == 200:
                return response.json().get("response", "")
            return f"Erreur: {response.status_code}"
        
        except Exception as e:
            return f"Erreur de connexion: {e}"
    
    def generate_contract(self, description: str) -> str:
        """Genere un contrat Solidity"""
        prompt = SOLIDITY_PROMPTS["contract_generation"].format(description=description)
        response = self.generate(prompt)
        return extract_solidity_code(response)


# Initialiser l'assistant local
local_assistant = LocalLLMAssistant()
print(f"LLM local disponible: {local_assistant.available}")
print(f"Endpoint: {local_assistant.endpoint}")
print(f"Modele: {local_assistant.model}")

In [ ]:
# Demo LLM local (si disponible)
print("=== TEST LLM LOCAL ===")

if local_assistant.available:
    code = local_assistant.generate_contract("Un simple contrat HelloWorld")
    print("Contrat genere localement:")
    print(code)
else:
    print("LLM local non disponible.")
    print("\nPour utiliser un LLM local:")
    print("  1. Installer Ollama: https://ollama.ai")
    print("  2. Telecharger le modele: ollama pull qwen2.5-coder:7b")
    print("  3. Demarrer Ollama: ollama serve")

### Comparaison API Cloud vs Local

| Aspect | API Cloud (Claude/GPT) | LLM Local (Qwen) |
|--------|----------------------|------------------|
| **Qualite code** | Excellente | Bonne |
| **Latence** | 1-5 secondes | 5-30 secondes |
| **Cout** | Par token | Gratuit |
| **Confidentialite** | Donnees envoyees | Tout local |
| **Disponibilite** | 99.9% | Depend du hardware |
| **Setup** | Cle API | Installation locale |

## 8. Resume et Bonnes Pratiques

### Ce que nous avons appris

1. **Prompting pour Solidity** : Structure et precision sont essentielles
2. **Generation de contrats** : LLMs peuvent produire du code fonctionnel rapidement
3. **Audit de securite** : Detection de vulnerabilites assistee par IA
4. **Documentation NatSpec** : Generation automatique des commentaires
5. **API Claude** : Integration avec Anthropic pour des resultats optimaux
6. **LLM Local** : Alternative open-source avec Qwen/Ollama

### Checklist avant deploiement

- [ ] Code genere par LLM audite manuellement
- [ ] Tests unitaires ecrits avec Foundry
- [ ] Fuzz testing effectue
- [ ] Audit formel avec Certora (si critique)
- [ ] Documentation NatSpec complete
- [ ] Code review par un expert humain

## Exercices

### Exercice 1 : Generer un Token ERC-20

Utilisez l'assistant LLM pour generer un token ERC-20 complet avec :
- Nom, symbole et decimales configurables
- Fonctions mint et burn (owner only)
- Events Transfer et Approval
- Documentation NatSpec complete

Puis auditez le code genere pour identifier les eventuels problemes.

In [ ]:
# Exercice 1 - Espace de travail
# TODO: Generer un ERC-20 avec l'assistant LLM
# TODO: Auditer le code genere

# Votre code ici...
erc20_description = """
# Votre description du token ERC-20 ici
"""

# Generation
# result = assistant.full_workflow(erc20_description)
# print(result)

### Exercice 2 : Workflow de Securite

Creez une fonction qui :
1. Genere un contrat a partir d'une description
2. Audite le code pour les vulnerabilites
3. Si des problemes sont trouves, demande au LLM de corriger
4. Repeat jusqu'a ce que l'audit soit propre ou max iterations atteint

Testez avec un contrat de voting simple.

In [ ]:
# Exercice 2 - Espace de travail
def secure_generation_loop(description: str, max_iterations: int = 3):
    """
    Boucle de generation securisee avec auto-correction.
    
    1. Generer le contrat
    2. Auditer
    3. Si vulnerabilites -> corriger et recommencer
    4. Retourner le contrat final et le rapport d'audit
    """
    # TODO: Implementer la logique
    pass

# Test avec un contrat de voting
# voting_description = "Un contrat de voting ou les utilisateurs peuvent..."
# result = secure_generation_loop(voting_description)
# print(result)

## Ressources Complementaires

### Documentation
- [Anthropic API Docs](https://docs.anthropic.com/)
- [OpenAI API Docs](https://platform.openai.com/docs/)
- [Ollama](https://ollama.ai/) - LLM local
- [Solidity NatSpec](https://docs.soliditylang.org/en/latest/natspec-format.html)

### Outils
- [Cursor](https://cursor.sh/) - IDE avec LLM integre
- [GitHub Copilot](https://github.com/features/copilot) - Autocompletion IA
- [Slither](https://github.com/crytic/slither) - Analyse statique Solidity

---

**Navigation** : [Index](../README.md) | [<< Account Abstraction](SC-8-Account-Abstraction.ipynb) | [Foundry Basics >>](../03-Foundry-Testing/SC-9-Foundry-Basics.ipynb)